In [ ]:
!pip -q install sentence-transformers

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

BASE = Path("/")
INPUT_DIR = BASE / ""
OUTPUT_DIR = BASE / ""

FILES = {
    "email": "email_labeled.csv",
    "http": "http_labeled.csv",
    "file": "file_labeled.csv",
}

MODEL_NAME = ""
BATCH_SIZE = 256

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"입력 폴더가 없습니다: {INPUT_DIR}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
DIM = model.get_embedding_dimension()

print(f"Device={device}, embedding_dim={DIM}")

for modality, filename in FILES.items():
    input_path = INPUT_DIR / filename
    modality_dir = OUTPUT_DIR / modality
    modality_dir.mkdir(parents=True, exist_ok=True)

    if not input_path.exists():
        raise FileNotFoundError(f"[{modality}] 파일이 없습니다: {input_path}")

    # content 임베딩에 필요한 user/date/content만 읽음
    df = pd.read_csv(input_path, usecols=["user", "date", "content"], low_memory=False)

    if df.empty:
        raise ValueError(f"[{modality}] 데이터가 0행입니다.")

    if df["user"].isna().any():
        raise ValueError(f"[{modality}] user 결측값이 존재합니다.")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if df["date"].isna().any():
        raise ValueError(f"[{modality}] date parsing 실패 행이 존재합니다.")

    # 이후 user-day pooling 시에도 반드시 같은 user/date 정렬을 사용해야 함
    df = df.sort_values(["user", "date"], kind="stable").reset_index(drop=True)

    total_rows = 0
    print(f"\n[{modality.upper()}] users={df['user'].nunique():,}, rows={len(df):,}")

    for user, user_df in df.groupby("user", sort=False):
        texts = user_df["content"].fillna("").astype(str).str.strip()
        has_text = texts.ne("")

        embeddings = np.zeros((len(user_df), DIM), dtype=np.float32)

        if has_text.any():
            embeddings[has_text.to_numpy()] = model.encode(
                texts[has_text].tolist(),
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False
            ).astype(np.float32)

        output_path = modality_dir / f"{modality}_{user}_embedding.pkl"

        # PKL에는 embedding NumPy 배열만 저장
        with open(output_path, "wb") as f:
            pickle.dump(embeddings, f, protocol=pickle.HIGHEST_PROTOCOL)

        # 저장 검증
        with open(output_path, "rb") as f:
            saved = pickle.load(f)

        if saved.shape != (len(user_df), DIM) or not np.isfinite(saved).all():
            raise RuntimeError(f"[{modality}/{user}] PKL 저장 검증 실패")

        total_rows += len(user_df)

    if total_rows != len(df):
        raise RuntimeError(f"[{modality}] 저장 행 수 불일치")

    print(f"[{modality.upper()}] 저장 완료: {modality_dir}")

print("\n모든 사용자별 content embedding PKL 저장 완료.")